# Microfossil Abundance Plot

This notebook visualizes the distribution and abundance of microfossils across different depths from the BOEM paleontology data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

In [ ]:
# Load the paleontology data
data_path = Path('data/2026-09-05/paleo_picks.csv')
df = pd.read_csv(data_path)

# Display first few rows
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Clean and prepare data
# Filter out rows with missing depth or age information
df_clean = df.dropna(subset=['paleo_sample_md', 'paleo_age'])

# Count microfossil occurrences by depth
abundance_by_depth = df_clean.groupby('paleo_sample_md').size().reset_index(name='abundance')
abundance_by_depth = abundance_by_depth.sort_values('paleo_sample_md')

print(f"Total records: {len(df_clean)}")
print(f"Unique depths: {len(abundance_by_depth)}")
print(f"\nDepth range: {abundance_by_depth['paleo_sample_md'].min():.0f} - {abundance_by_depth['paleo_sample_md'].max():.0f} meters")
print(f"\nAbundance statistics:")
print(abundance_by_depth['abundance'].describe())

In [ ]:
# Figure 1: Vertical Abundance Plot
fig, ax = plt.subplots(figsize=(10, 12))

# Create the abundance plot with depth on y-axis (inverted so shallow is at top)
bars = ax.barh(abundance_by_depth['paleo_sample_md'], abundance_by_depth['abundance'], 
                height=10, color='#2E86AB', alpha=0.8, edgecolor='#1B4965', linewidth=0.5)

ax.set_ylabel('Depth (meters TVD)', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Microfossil Records', fontsize=12, fontweight='bold')
ax.set_title('Microfossil Abundance Distribution by Depth\nUS-BOEM Paleontology Data', 
             fontsize=14, fontweight='bold', pad=20)

# Invert y-axis so shallow depths are at top
ax.invert_yaxis()

# Add grid for better readability
ax.grid(axis='x', alpha=0.3, linestyle='--')

# Format layout
plt.tight_layout()
plt.savefig('figures/microfossil_abundance_vertical.png', dpi=300, bbox_inches='tight')
plt.show()

print("Vertical abundance plot saved to: figures/microfossil_abundance_vertical.png")

In [ ]:
# Figure 2: Scatter plot with better depth resolution
fig, ax = plt.subplots(figsize=(10, 12))

scatter = ax.scatter(abundance_by_depth['abundance'], 
                     abundance_by_depth['paleo_sample_md'],
                     s=abundance_by_depth['abundance']*30,  # Size proportional to abundance
                     c=abundance_by_depth['abundance'],
                     cmap='Blues', alpha=0.6, edgecolors='#1B4965', linewidth=1)

ax.set_xlabel('Number of Microfossil Records', fontsize=12, fontweight='bold')
ax.set_ylabel('Depth (meters TVD)', fontsize=12, fontweight='bold')
ax.set_title('Microfossil Abundance by Depth (Bubble Plot)\nBubble size proportional to abundance', 
             fontsize=14, fontweight='bold', pad=20)

# Invert y-axis so shallow depths are at top
ax.invert_yaxis()

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Abundance', fontsize=10)

ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('figures/microfossil_abundance_bubble.png', dpi=300, bbox_inches='tight')
plt.show()

print("Bubble plot saved to: figures/microfossil_abundance_bubble.png")

In [ ]:
# Figure 3: Species/Age Distribution by Depth
# Get top species by frequency
top_species = df_clean['paleo_age'].value_counts().head(10).index.tolist()

# Create a subset with top species
df_top_species = df_clean[df_clean['paleo_age'].isin(top_species)].copy()

# Count by depth and species
species_abundance = df_top_species.groupby(['paleo_sample_md', 'paleo_age']).size().reset_index(name='count')

# Create pivot table for stacked bar
pivot_data = species_abundance.pivot(index='paleo_sample_md', columns='paleo_age', values='count').fillna(0)

fig, ax = plt.subplots(figsize=(12, 12))

# Define color palette
colors = sns.color_palette('husl', len(top_species))

# Create stacked horizontal bar chart
pivot_data.plot(kind='barh', stacked=True, ax=ax, color=colors, width=0.7, edgecolor='white', linewidth=0.5)

ax.set_ylabel('Depth (meters TVD)', fontsize=12, fontweight='bold')
ax.set_xlabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Microfossil Species Distribution by Depth\nTop 10 Species', 
             fontsize=14, fontweight='bold', pad=20)

# Customize legend
ax.legend(title='Species/Age', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, title_fontsize=10)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('figures/microfossil_species_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Species distribution plot saved to: figures/microfossil_species_distribution.png")

In [ ]:
# Figure 4: Range Through Chart (First and Last Occurrence)
# Calculate stratigraphic range for each species
species_ranges = df_clean.groupby('paleo_age')['paleo_sample_md'].agg(['min', 'max', 'count']).reset_index()
species_ranges.columns = ['paleo_age', 'first_occurrence', 'last_occurrence', 'count']
species_ranges = species_ranges.sort_values('last_occurrence')

# Filter for species with sufficient records
species_ranges = species_ranges[species_ranges['count'] >= 2].head(15)

fig, ax = plt.subplots(figsize=(12, 10))

# Plot ranges
y_pos = np.arange(len(species_ranges))
for i, (idx, row) in enumerate(species_ranges.iterrows()):
    ax.plot([row['first_occurrence'], row['last_occurrence']], [i, i], 
           'o-', linewidth=3, markersize=8, color='#2E86AB', alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels([name[:50] for name in species_ranges['paleo_age']], fontsize=9)
ax.set_xlabel('Depth (meters TVD)', fontsize=12, fontweight='bold')
ax.set_title('Microfossil Stratigraphic Range Chart\nFirst and Last Occurrence by Species (15 species, n≥2)', 
             fontsize=14, fontweight='bold', pad=20)
ax.invert_xaxis()
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('figures/microfossil_range_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print("Range chart saved to: figures/microfossil_range_chart.png")

In [ ]:
# Summary Statistics
print("\n" + "="*60)
print("MICROFOSSIL ABUNDANCE ANALYSIS SUMMARY")
print("="*60)

print(f"\nTotal records analyzed: {len(df_clean):,}")
print(f"Total wells: {df_clean['api_well_number'].nunique()}")
print(f"Total unique species/ages identified: {df_clean['paleo_age'].nunique()}")
print(f"Unique sample depths: {df_clean['paleo_sample_md'].nunique()}")

print(f"\nDepth Statistics:")
print(f"  Shallowest sample: {df_clean['paleo_sample_md'].min():.1f} meters")
print(f"  Deepest sample: {df_clean['paleo_sample_md'].max():.1f} meters")
print(f"  Range: {df_clean['paleo_sample_md'].max() - df_clean['paleo_sample_md'].min():.1f} meters")

print(f"\nAbundance Statistics:")
print(f"  Mean records per depth: {abundance_by_depth['abundance'].mean():.1f}")
print(f"  Median records per depth: {abundance_by_depth['abundance'].median():.1f}")
print(f"  Max records at any depth: {abundance_by_depth['abundance'].max():.0f}")
print(f"  Min records at any depth: {abundance_by_depth['abundance'].min():.0f}")

print(f"\nTop 10 Most Common Species/Ages:")
top_10 = df_clean['paleo_age'].value_counts().head(10)
for species, count in top_10.items():
    print(f"  {count:4d} - {species[:60]}")

print("\n" + "="*60)